# Free GPU worker

One notebook for every free GPU: **Google Colab, Kaggle, or any Jupyter box with an NVIDIA T4 or newer.**
It runs the next slice of a job from the shared hub and stops cleanly before the session limit.
Whatever it finishes is saved, and the next worker, on any provider, continues from there.

**Once per provider**
- Colab: Runtime → Change runtime type → T4 GPU. Add a secret `HF_TOKEN` (key icon on the left).
- Kaggle: Settings → Accelerator → GPU T4 x2, Internet on. Add-ons → Secrets → `HF_TOKEN`.
- Anywhere else: `export HF_TOKEN=...` before starting Jupyter.

Then set `HUB` and `JOB` below and **Run all**.

In [ ]:
HUB = "hf:YOUR_HF_USERNAME/anarkali-work"   # private Hugging Face dataset repo holding the job
JOB = "typed-v2-relabel"
REPO = "https://github.com/ToufiqQureshi/anarkali"
REF = "main"       # branch to run
MAX_HOURS = 10     # stop before the free session ends (Colab and Kaggle cut off around 12 h)

In [ ]:
import subprocess, sys
subprocess.run(["git", "clone", "--depth", "1", "--branch", REF, REPO, "anarkali"], check=False)
subprocess.run(["git", "-C", "anarkali", "pull", "--ff-only"], check=False)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "vllm", "huggingface_hub"])

In [ ]:
# Colab secrets are only readable from the notebook itself, so hand HF_TOKEN to the worker here.
import os
sys.path.insert(0, "anarkali/freegpu")
from hub import find_secret
os.environ["HF_TOKEN"] = find_secret("HF_TOKEN") or ""
assert os.environ["HF_TOKEN"], "add a secret named HF_TOKEN first (see the top of this notebook)"
subprocess.check_call([sys.executable, "anarkali/freegpu/worker.py", "--hub", HUB, "--job", JOB,
                       "--max-hours", str(MAX_HOURS)])